# 02 - Feature Engineering and Feature Selection

This notebook builds advanced features and selects the production feature set.


In [1]:
# Cell 0: Setup and load dataset
from __future__ import annotations

import json
from datetime import UTC, datetime
from pathlib import Path
import sys


import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance

ROOT = Path.cwd()
for p in [ROOT, *ROOT.parents]:
    if (p / 'ml' / 'pipelines' / 'churn_train.py').exists():
        ROOT = p
        break

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from ml.pipelines.churn_train import split_by_snapshot

DATASET_PATH = ROOT / 'ml' / 'data' / 'churn_training_dataset.csv'
assert DATASET_PATH.exists(), 'Run 01_advanced_eda.ipynb first.'

df = pd.read_csv(DATASET_PATH)
df['snapshot_date'] = pd.to_datetime(df['snapshot_date'], utc=True)
df['will_order_next_30d'] = df['will_order_next_30d'].astype(int)

df = df.sort_values(['snapshot_date', 'user_id']).reset_index(drop=True)

print('Loaded dataset:', df.shape)
print('Snapshot count:', df['snapshot_date'].nunique())
print('Target rate:', round(df['will_order_next_30d'].mean(), 4))



Loaded dataset: (862, 25)
Snapshot count: 12
Target rate: 0.5162


In [2]:
# Cell 1: EDA-aware baseline checks
snapshot_profile = (
    df.groupby('snapshot_date', as_index=False)
      .agg(
          users=('user_id', 'nunique'),
          target_rate=('will_order_next_30d', 'mean'),
          avg_orders_30d=('orders_30d', 'mean'),
          avg_revenue_lookback=('revenue_lookback', 'mean'),
      )
      .sort_values('snapshot_date')
)

non_features = {'user_id', 'snapshot_date', 'will_order_next_30d'}
base_numeric = [
    c for c in df.columns
    if c not in non_features and pd.api.types.is_numeric_dtype(df[c])
]

base_missing = df[base_numeric].isna().mean().sort_values(ascending=False)
base_zero_var = [c for c in base_numeric if df[c].nunique(dropna=False) <= 1]

print('Base numeric features:', len(base_numeric))
print('Zero-variance base features:', base_zero_var)
print('Top missing ratios:')
display(base_missing.head(10).to_frame('missing_ratio'))
print('Snapshot profile:')
display(snapshot_profile)



Base numeric features: 22
Zero-variance base features: ['promo_order_ratio_lookback', 'avg_rating_lifetime', 'review_count_lifetime']
Top missing ratios:


,missing_ratio
orders_lookback,0.0
orders_30d,0.0
avg_rating_lifetime,0.0
avg_distinct_products_per_order_lookback,0.0
avg_items_per_order_lookback,0.0
max_gap_days_lookback,0.0
std_gap_days_lookback,0.0
avg_gap_days_lookback,0.0
revenue_lifetime,0.0
order_count_lifetime,0.0


Snapshot profile:


,snapshot_date,users,target_rate,avg_orders_30d,avg_revenue_lookback
0,2025-11-20 00:00:00+00:00,34,0.647059,1.941176,205.517059
1,2025-11-27 00:00:00+00:00,37,0.594595,2.054054,240.257297
2,2025-12-04 00:00:00+00:00,47,0.595745,1.936170,241.157660
3,2025-12-11 00:00:00+00:00,52,0.576923,2.000000,263.762885
4,2025-12-18 00:00:00+00:00,55,0.618182,2.236364,305.617636
5,2025-12-25 00:00:00+00:00,61,0.590164,2.196721,314.816557
6,2026-01-01 00:00:00+00:00,69,0.550725,2.028986,317.840580
7,2026-01-08 00:00:00+00:00,76,0.539474,2.013158,330.094737
8,2026-01-15 00:00:00+00:00,86,0.523256,2.093023,348.930116
9,2026-01-22 00:00:00+00:00,98,0.489796,2.173469,364.033878


In [3]:
# Cell 2: Advanced feature engineering (time-series + cohort + anomaly + segmentation)
feat = df.copy().sort_values(['user_id', 'snapshot_date']).reset_index(drop=True)
EPS = 1e-6


def _safe_div(num: pd.Series, den: pd.Series) -> pd.Series:
    return num / (den.replace(0, np.nan) + EPS)


# Core interactions (RFM style)
feat['orders_30d_per_age'] = _safe_div(feat['orders_30d'], feat['customer_age_days'] + 1.0)
feat['revenue_per_order_lifetime'] = _safe_div(feat['revenue_lifetime'], feat['order_count_lifetime'] + 1.0)
feat['recency_gap_ratio'] = _safe_div(feat['days_since_last_order'], feat['avg_gap_days_lookback'] + 1.0)
feat['value_gap_ratio'] = _safe_div(feat['avg_order_value_lookback'], feat['avg_gap_days_lookback'] + 1.0)
feat['items_x_aov'] = feat['avg_items_per_order_lookback'] * feat['avg_order_value_lookback']
feat['distinct_x_orders'] = feat['avg_distinct_products_per_order_lookback'] * feat['orders_30d']

# Gap-anomaly metrics from advanced EDA
feat['gap_multiplier'] = _safe_div(feat['days_since_last_order'], feat['avg_gap_days_lookback'] + 1.0)
feat['is_gap_anomaly_2x'] = (feat['gap_multiplier'] >= 2.0).astype(int)
feat['is_gap_anomaly_3x'] = (feat['gap_multiplier'] >= 3.0).astype(int)

# Stabilized transforms for skewed positive features
for col in [
    'revenue_lookback',
    'revenue_lifetime',
    'days_since_last_order',
    'avg_gap_days_lookback',
    'std_order_value_lookback',
    'gap_multiplier',
]:
    if col in feat.columns:
        feat[f'log1p_{col}'] = np.log1p(np.clip(feat[col], a_min=0, a_max=None))

# Time-series lag and rolling features per user (chapter 3 style)
lag_base_cols = [
    'orders_30d',
    'revenue_lookback',
    'days_since_last_order',
    'avg_order_value_lookback',
]
for col in lag_base_cols:
    if col not in feat.columns:
        continue
    feat[f'{col}_lag1'] = feat.groupby('user_id')[col].shift(1)
    feat[f'{col}_diff1'] = feat[col] - feat[f'{col}_lag1']
    feat[f'{col}_pct_change1'] = _safe_div(feat[f'{col}_diff1'], feat[f'{col}_lag1'].abs() + 1.0)
    feat[f'{col}_roll3_mean_lagged'] = feat.groupby('user_id')[col].transform(
        lambda s: s.shift(1).rolling(window=3, min_periods=1).mean()
    )
    feat[f'{col}_vs_roll3'] = feat[col] - feat[f'{col}_roll3_mean_lagged']

# Snapshot-level z-scores (chapter 6 style anomaly standardization)
anom_cols = ['orders_30d', 'revenue_lookback', 'days_since_last_order', 'std_order_value_lookback']
for col in anom_cols:
    if col not in feat.columns:
        continue
    mu = feat.groupby('snapshot_date')[col].transform('mean')
    sd = feat.groupby('snapshot_date')[col].transform('std').replace(0, np.nan)
    z = ((feat[col] - mu) / sd).fillna(0.0)
    feat[f'{col}_zscore_snapshot'] = z
    feat[f'{col}_is_anom_z2'] = (z.abs() >= 2.0).astype(int)

z_cols = [c for c in feat.columns if c.endswith('_zscore_snapshot')]
if z_cols:
    feat['anomaly_score'] = feat[z_cols].abs().sum(axis=1) + feat['is_gap_anomaly_2x']
else:
    feat['anomaly_score'] = feat['is_gap_anomaly_2x'].astype(float)

# Cohort dynamics (chapter 4 style)
feat['cohort_start_date'] = feat.groupby('user_id')['snapshot_date'].transform('min')
ordered_snapshots = sorted(feat['snapshot_date'].dropna().unique().tolist())
snapshot_index = {k: i for i, k in enumerate(ordered_snapshots)}
feat['snapshot_idx'] = feat['snapshot_date'].map(snapshot_index).astype(int)
feat['cohort_idx'] = feat['cohort_start_date'].map(snapshot_index).astype(int)
feat['cohort_period'] = feat['snapshot_idx'] - feat['cohort_idx']

cohort_sizes = feat.groupby('cohort_start_date')['user_id'].nunique().rename('cohort_size')
feat = feat.merge(cohort_sizes, on='cohort_start_date', how='left')

cohort_active = (
    feat[feat['orders_30d'] > 0]
    .groupby(['cohort_start_date', 'cohort_period'])['user_id']
    .nunique()
    .rename('cohort_active_users')
)
feat = feat.merge(cohort_active, on=['cohort_start_date', 'cohort_period'], how='left')
feat['cohort_active_users'] = feat['cohort_active_users'].fillna(0)
feat['cohort_retention_rate'] = _safe_div(feat['cohort_active_users'], feat['cohort_size'])

cohort_rev0 = (
    feat[feat['cohort_period'] == 0]
    .groupby('cohort_start_date')['revenue_lookback']
    .median()
    .rename('cohort_rev0')
)
feat = feat.merge(cohort_rev0, on='cohort_start_date', how='left')
feat['cohort_revenue_index'] = _safe_div(feat['revenue_lookback'], feat['cohort_rev0'] + 1.0)

# Snapshot seasonal context (chapter 3)
week = feat['snapshot_date'].dt.isocalendar().week.astype(int)
month = feat['snapshot_date'].dt.month.astype(int)
feat['snapshot_weekofyear'] = week
feat['snapshot_month'] = month
feat['snapshot_week_sin'] = np.sin(2 * np.pi * week / 52.0)
feat['snapshot_week_cos'] = np.cos(2 * np.pi * week / 52.0)
feat['snapshot_month_sin'] = np.sin(2 * np.pi * month / 12.0)
feat['snapshot_month_cos'] = np.cos(2 * np.pi * month / 12.0)

# Experiment-ready business segments (chapter 7 mindset)
value_cut = feat['revenue_lifetime'].quantile(0.75)
eng_cut = feat['orders_30d'].quantile(0.75)
feat['is_high_value'] = (feat['revenue_lifetime'] >= value_cut).astype(int)
feat['is_high_engagement'] = (feat['orders_30d'] >= eng_cut).astype(int)
feat['is_high_value_high_risk'] = (
    (feat['is_high_value'] == 1)
    & ((feat['is_gap_anomaly_2x'] == 1) | (feat.get('days_since_last_order_zscore_snapshot', 0) > 1.5))
).astype(int)

# Winsorized copies for robust modeling (chapter 8 dimensionality/risk control)
for col in [
    'revenue_lookback',
    'revenue_lifetime',
    'days_since_last_order',
    'std_order_value_lookback',
    'gap_multiplier',
    'anomaly_score',
]:
    if col in feat.columns:
        lo, hi = feat[col].quantile([0.01, 0.99])
        feat[f'{col}_clip'] = feat[col].clip(lo, hi)

engineered_path = ROOT / 'ml' / 'data' / 'churn_training_features.csv'
feat.to_csv(engineered_path, index=False)

print('Engineered dataset shape:', feat.shape)
print('Saved engineered features to:', engineered_path)



Engineered dataset shape: (862, 93)
Saved engineered features to: /Users/deliorincon/Desktop/Sliceiq/ml/data/churn_training_features.csv


In [4]:
# Cell 3: Feature quality checks and candidate pool
non_features = {'user_id', 'snapshot_date', 'cohort_start_date', 'will_order_next_30d'}
all_numeric_features = [
    c for c in feat.columns
    if c not in non_features and pd.api.types.is_numeric_dtype(feat[c])
]

missing_ratio = feat[all_numeric_features].isna().mean().sort_values(ascending=False)
low_variance = [c for c in all_numeric_features if feat[c].nunique(dropna=False) <= 1]

candidate_features = [c for c in all_numeric_features if c not in low_variance]

print('All numeric features:', len(all_numeric_features))
print('Low-variance dropped:', len(low_variance))
print('Candidate features:', len(candidate_features))

if low_variance:
    print('Low-variance columns:', low_variance)

display(missing_ratio.head(20).to_frame('missing_ratio'))



All numeric features: 89
Low-variance dropped: 3
Candidate features: 86
Low-variance columns: ['promo_order_ratio_lookback', 'avg_rating_lifetime', 'review_count_lifetime']


,missing_ratio
revenue_lookback_pct_change1,0.156613
orders_30d_vs_roll3,0.156613
avg_order_value_lookback_roll3_mean_lagged,0.156613
avg_order_value_lookback_pct_change1,0.156613
avg_order_value_lookback_diff1,0.156613
orders_30d_lag1,0.156613
orders_30d_diff1,0.156613
orders_30d_pct_change1,0.156613
orders_30d_roll3_mean_lagged,0.156613
revenue_lookback_lag1,0.156613


In [5]:
# Cell 4: Time-based split + collinearity pruning
train_df, val_df, test_df = split_by_snapshot(feat)

X_train = train_df[candidate_features]
y_train = train_df['will_order_next_30d']
X_val = val_df[candidate_features]
y_val = val_df['will_order_next_30d']

corr = X_train.corr(numeric_only=True).abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

high_corr_pairs = []
for col in upper.columns:
    for row, value in upper[col].dropna().items():
        if value >= 0.98:
            high_corr_pairs.append((row, col, float(value)))

# prune near-duplicates only (keep signal, remove redundant copies)
prune_threshold = 0.995
to_drop = [c for c in upper.columns if (upper[c] > prune_threshold).any()]
pruned_features = [c for c in candidate_features if c not in to_drop]

print('train:', train_df.shape, '| val:', val_df.shape, '| test:', test_df.shape)
print('High-correlation pairs (|corr| >= 0.98):', len(high_corr_pairs))
print('Pruned for near-duplication (|corr| > 0.995):', len(to_drop))
print('Remaining features for ranking:', len(pruned_features))

display(pd.DataFrame(high_corr_pairs, columns=['feature_a', 'feature_b', 'abs_corr']).head(25))



train: (355, 93) | val: (162, 93) | test: (345, 93)
High-correlation pairs (|corr| >= 0.98): 25
Pruned for near-duplication (|corr| > 0.995): 13
Remaining features for ranking: 73


,feature_a,feature_b,abs_corr
0,orders_lookback,orders_90d,0.996131
1,orders_lookback,order_count_lifetime,1.000000
2,orders_90d,order_count_lifetime,0.996131
3,revenue_lookback,revenue_lifetime,1.000000
4,recency_gap_ratio,gap_multiplier,1.000000
5,log1p_revenue_lookback,log1p_revenue_lifetime,1.000000
6,revenue_lookback_lag1,revenue_lookback_roll3_mean_lagged,0.984506
7,avg_order_value_lookback_lag1,avg_order_value_lookback_roll3_mean_lagged,0.991862
8,orders_30d,orders_30d_zscore_snapshot,0.984806
9,days_since_last_order,days_since_last_order_zscore_snapshot,0.981969


In [6]:
# Cell 5: Mutual information ranking
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train[pruned_features])
X_val_imp = imputer.transform(X_val[pruned_features])

mi_scores = mutual_info_classif(X_train_imp, y_train, random_state=42)
mi_rank = (
    pd.DataFrame({'feature': pruned_features, 'mi': mi_scores})
    .sort_values('mi', ascending=False)
    .reset_index(drop=True)
)

display(mi_rank.head(40))



,feature,mi
0,log1p_revenue_lookback,0.458402
1,revenue_lookback,0.453890
2,avg_order_value_lookback,0.443770
3,items_x_aov,0.442620
4,max_gap_days_lookback,0.437596
5,revenue_per_order_lifetime,0.426679
6,log1p_avg_gap_days_lookback,0.421219
7,cohort_revenue_index,0.421093
8,avg_gap_days_lookback,0.410179
9,value_gap_ratio,0.404977


In [8]:
# Cell 6: Permutation importance + stability diagnostics
candidate_features = mi_rank.head(min(45, len(mi_rank)))['feature'].tolist()
idx = [pruned_features.index(c) for c in candidate_features]

rf = RandomForestClassifier(
    n_estimators=600,
    max_depth=10,
    min_samples_leaf=6,
    random_state=42,
    n_jobs=1,
    class_weight='balanced_subsample',
)
rf.fit(X_train_imp[:, idx], y_train)

perm = permutation_importance(
    rf,
    X_val_imp[:, idx],
    y_val,
    n_repeats=10,
    random_state=42,
    scoring='average_precision',
    n_jobs=1,
)

perm_rank = (
    pd.DataFrame({'feature': candidate_features, 'perm_importance': perm.importances_mean})
    .sort_values('perm_importance', ascending=False)
    .reset_index(drop=True)
)

# Stability: per-snapshot corr(feature, target) in training snapshots.
stability_rows = []
for feature in candidate_features:
    corrs = []
    for _, chunk in train_df[['snapshot_date', feature, 'will_order_next_30d']].groupby('snapshot_date'):
        if chunk[feature].nunique(dropna=True) <= 1:
            continue
        if chunk['will_order_next_30d'].nunique(dropna=True) <= 1:
            continue
        c = chunk[feature].corr(chunk['will_order_next_30d'])
        if pd.notna(c):
            corrs.append(float(c))
    if corrs:
        mean_abs = float(np.mean(np.abs(corrs)))
        std_corr = float(np.std(corrs))
    else:
        mean_abs = 0.0
        std_corr = np.nan
    stability_rows.append(
        {
            'feature': feature,
            'snapshot_corr_mean_abs': mean_abs,
            'snapshot_corr_std': std_corr,
        }
    )

stability = pd.DataFrame(stability_rows)
stability['stability_score'] = stability['snapshot_corr_mean_abs'] / (stability['snapshot_corr_std'].fillna(1.0) + 0.05)

ranked = perm_rank.merge(stability, on='feature', how='left')
ranked['perm_rank_pct'] = ranked['perm_importance'].rank(pct=True)
ranked['final_score'] = ranked['perm_rank_pct'] + 0.25 * ranked['stability_score'].fillna(0)
ranked = ranked.sort_values('final_score', ascending=False).reset_index(drop=True)

print('Top permutation features:')
display(perm_rank.head(25))
print('Top combined (importance + stability) features:')
display(ranked.head(25))




Top permutation features:


,feature,perm_importance
0,dinner_order_ratio_lookback,0.005559
1,max_gap_days_lookback,0.005214
2,orders_30d_per_age,0.003330
3,std_gap_days_lookback,0.002434
4,revenue_lookback_zscore_snapshot,0.002104
5,log1p_avg_gap_days_lookback,0.001983
6,avg_gap_days_lookback,0.001815
7,log1p_std_order_value_lookback,0.001744
8,days_since_last_order_vs_roll3,0.001630
9,recency_gap_ratio,0.001598


Top combined (importance + stability) features:


,feature,perm_importance,snapshot_corr_mean_abs,snapshot_corr_std,stability_score,perm_rank_pct,final_score
0,log1p_std_order_value_lookback,0.001744,0.797343,0.080807,6.095557,0.844444,2.368334
1,recency_gap_ratio,0.001598,0.625984,0.053140,6.069243,0.800000,2.317311
2,orders_60d,0.001378,0.596414,0.061926,5.328661,0.711111,2.043276
3,log1p_revenue_lookback,0.001336,0.694462,0.078631,5.398868,0.688889,2.038606
4,std_order_value_lookback_clip,0.001429,0.717423,0.099076,4.812458,0.733333,1.936448
5,std_order_value_lookback,0.001438,0.710218,0.102848,4.646573,0.755556,1.917199
6,orders_30d_per_age,0.003330,0.448007,0.067443,3.814662,0.955556,1.909221
7,orders_lookback,0.001001,0.577089,0.063561,5.081746,0.622222,1.892659
8,revenue_lookback_zscore_snapshot,0.002104,0.581453,0.099026,3.901683,0.911111,1.886532
9,log1p_avg_gap_days_lookback,0.001983,0.658250,0.117541,3.928886,0.888889,1.871110


In [9]:
# Cell 7: Final feature set (coverage across feature themes)
def _theme_of(feature: str) -> str:
    if 'cohort_' in feature:
        return 'cohort'
    if 'zscore_snapshot' in feature or 'anomaly' in feature or 'gap_multiplier' in feature:
        return 'anomaly'
    if 'lag1' in feature or 'diff1' in feature or 'pct_change1' in feature or 'roll3' in feature or feature.startswith('snapshot_'):
        return 'time_series'
    if 'revenue' in feature or 'order_value' in feature or 'items' in feature or 'distinct' in feature:
        return 'value_mix'
    return 'core'

ranked = ranked.copy()
ranked['theme'] = ranked['feature'].map(_theme_of)

# Keep positive-permutation features; fallback to top rows.
positive = ranked[ranked['perm_importance'] > 0].copy()
if positive.empty:
    positive = ranked.head(20).copy()

selected = []
for theme in ['core', 'time_series', 'cohort', 'anomaly', 'value_mix']:
    theme_rows = positive[positive['theme'] == theme]
    if not theme_rows.empty:
        selected.append(theme_rows.iloc[0]['feature'])

for feature in positive['feature']:
    if feature not in selected:
        selected.append(feature)
    if len(selected) >= 24:
        break

if len(selected) < 12:
    for feature in ranked['feature']:
        if feature not in selected:
            selected.append(feature)
        if len(selected) >= 12:
            break

selected = selected[:24]

print('Selected feature count:', len(selected))
print('Theme distribution:')
display(pd.Series([_theme_of(f) for f in selected]).value_counts().to_frame('count'))
print(selected)



Selected feature count: 24
Theme distribution:


,count
core,10
value_mix,6
time_series,4
anomaly,3
cohort,1


['recency_gap_ratio', 'revenue_lookback_lag1', 'cohort_revenue_index', 'revenue_lookback_zscore_snapshot', 'log1p_std_order_value_lookback', 'orders_60d', 'log1p_revenue_lookback', 'std_order_value_lookback_clip', 'std_order_value_lookback', 'orders_30d_per_age', 'orders_lookback', 'log1p_avg_gap_days_lookback', 'log1p_gap_multiplier', 'value_gap_ratio', 'revenue_lookback', 'orders_30d_zscore_snapshot', 'max_gap_days_lookback', 'revenue_per_order_lifetime', 'orders_30d_roll3_mean_lagged', 'std_gap_days_lookback', 'orders_30d', 'orders_30d_lag1', 'avg_gap_days_lookback', 'days_since_last_order_vs_roll3']


In [10]:
# Cell 8: Save artifacts and diagnostics
models_dir = ROOT / 'ml' / 'models'
models_dir.mkdir(parents=True, exist_ok=True)

selected_out = models_dir / 'selected_features_notebook.json'
selected_out.write_text(json.dumps({'selected_features': selected}, indent=2), encoding='utf-8')

ranking_out = models_dir / 'feature_ranking_notebook.csv'
ranked.to_csv(ranking_out, index=False)

report = {
    'generated_at_utc': datetime.now(UTC).isoformat(),
    'input_dataset': str(DATASET_PATH),
    'engineered_dataset_rows': int(len(feat)),
    'engineered_dataset_cols': int(feat.shape[1]),
    'base_numeric_features': int(len(base_numeric)),
    'all_numeric_features': int(len(all_numeric_features)),
    'low_variance_dropped': low_variance,
    'near_duplicate_dropped': to_drop,
    'candidate_after_prune': int(len(pruned_features)),
    'selected_feature_count': int(len(selected)),
    'selected_features': selected,
}

report_out = models_dir / 'feature_engineering_report_notebook.json'
report_out.write_text(json.dumps(report, indent=2), encoding='utf-8')

print('Saved selected features:', selected_out)
print('Saved feature ranking:', ranking_out)
print('Saved engineering report:', report_out)

selected_corr = (
    feat[selected + ['will_order_next_30d']]
    .corr(numeric_only=True)['will_order_next_30d']
    .drop('will_order_next_30d')
    .sort_values(key=lambda s: s.abs(), ascending=False)
)

display(selected_corr.to_frame('corr_with_target'))



Saved selected features: /Users/deliorincon/Desktop/Sliceiq/ml/models/selected_features_notebook.json
Saved feature ranking: /Users/deliorincon/Desktop/Sliceiq/ml/models/feature_ranking_notebook.csv
Saved engineering report: /Users/deliorincon/Desktop/Sliceiq/ml/models/feature_engineering_report_notebook.json


,corr_with_target
log1p_std_order_value_lookback,0.904394
std_order_value_lookback_clip,0.842736
std_order_value_lookback,0.835730
log1p_avg_gap_days_lookback,0.811150
log1p_gap_multiplier,-0.780060
log1p_revenue_lookback,0.777743
revenue_lookback_zscore_snapshot,0.693837
orders_60d,0.690221
orders_lookback,0.678294
revenue_lookback,0.674080


## Next Notebook

Proceed to `03_training_and_evaluation.ipynb`.
